# 17. Querying Everything, and Provenance

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `16_draft_companion_tools.ipynb` first.

Every chapter in this tutorial wrote into one shared DuckDB database. This closing notebook covers querying that database directly, the low-level helpers every `fetch_and_write_*` function is built on, and how to record exactly which code, config, and data produced a given result. Mirrors the [Querying and Provenance](https://nolmacdonald.github.io/nuclearff/tutorial/17_querying_and_provenance.html) docs page.

## Querying what you've built

Every table from Chapters 4 through 16 lives in one file. Query it directly with DuckDB, or with `read_table` to get a `polars.DataFrame` back:

In [1]:
import duckdb

from nuclearff.duckdb_io import read_table

db_path = "./demo/data/cache/nuclearff.duckdb"

with duckdb.connect(db_path, read_only=True) as conn:
    conn.sql("SHOW TABLES").show()

┌─────────────────────────────┐
│            name             │
│           varchar           │
├─────────────────────────────┤
│ player_id_map               │
│ sleeper_draft_picks         │
│ sleeper_league_configs      │
│ sleeper_leagues             │
│ sleeper_matchups            │
│ sleeper_players             │
│ sleeper_players_meta        │
│ sleeper_playoff_matches     │
│ sleeper_projections         │
│ sleeper_roster_names        │
│ sleeper_roster_players      │
│ sleeper_standings           │
│ sleeper_transaction_players │
│ sleeper_transactions        │
│ sleeper_user_avatars        │
└─────────────────────────────┘
            15 rows          



In [2]:
standings = read_table(db_path, "sleeper_standings")
standings.height

60

Every table is keyed by `league_id` (and usually `season`), so joining across them -- "which player did the eventual champion trade for" or "how many waiver claims did the team with the worst record make" -- is ordinary SQL, not custom Python for each question.

## The write primitives underneath

`nuclearff.duckdb_io` has two ways to write a table, and the difference between them mattered enough to cause (and then fix) a real bug in this project. `replace_table` drops and rewrites a table wholesale -- correct for a genuine single-source snapshot like the Sleeper player map. `merge_table` replaces only the rows a given call actually owns, by key, leaving every other row untouched:

> **Why this matters:** the cell below writes to a throwaway `merge_table_demo` table, not the real `sleeper_matchups` this notebook series has been building all along -- unlike the docs page's illustrative-only code block, this notebook actually executes it, and running `merge_table` with real `key_values` against `sleeper_matchups` here would delete every real matchup row for that league_id and replace it with one fabricated row. Same mechanism, safe target.

In [3]:
from nuclearff.duckdb_io import merge_table

merge_table(
    db_path,
    "merge_table_demo",
    create_table_sql="CREATE TABLE IF NOT EXISTS merge_table_demo (league_id VARCHAR, season INTEGER, week INTEGER, roster_id INTEGER, points DOUBLE)",
    columns=["league_id", "season", "week", "roster_id", "points"],
    rows=[["1367225133634191360", 2026, 2, 1, 118.4]],
    key_column="league_id",
    key_values=["1367225133634191360"],
)

1

**This distinction is not academic.** A real, previously-undetected bug in this exact project: every `fetch_and_write_*` function used to call `replace_table` with just one league's own rows -- which silently erased every *other* league's rows already in the same shared table, since `replace_table` always drops the whole table first. Confirmed live: fetching three unrelated leagues on the same account in sequence left only the last-fetched league's data surviving, with no error at any point. Every `fetch_and_write_*` function in Chapters 4, 10, 13, and 16 now uses `merge_table` instead -- if you're writing your own multi-league pipeline against a table more than one source populates, use `merge_table`, not `replace_table`.

> **Why this matters:** this bug is worth sitting with for a moment, because it's a genuinely easy trap: `replace_table` looks completely correct in every single-league test, since there's only ever one league's data in the table to begin with. It only breaks the moment a *second* league shares the same database file -- exactly the situation this dashboard-style tutorial notebook is in right now, writing multiple real leagues (`NUCLEARFF REDRAFT`, `Freeman Forever League`) into the same `./demo` cache across its chapters. If you ever write a new `fetch_and_write_*`-style function for this project, `merge_table` is the default to reach for, not `replace_table`.

## Provenance: recording how a result was built

A number in a report is only as trustworthy as your ability to reproduce it. `nuclearff.provenance` records exactly what produced a given artifact: the git commit, whether the working tree was clean, the config that was used, and hashes of anything that matters.

In [4]:
from nuclearff.config import default_config
from nuclearff.provenance import git_commit_sha, is_git_dirty, sha256_json

cfg = default_config()
print("commit:", git_commit_sha("."))
print("dirty: ", is_git_dirty("."))
print("config hash:", sha256_json(cfg.canonical_dict())[:16] + "...")

commit: d646fcee482c75d1dd16fe3e4cee95782bfcb703
dirty:  True
config hash: 2c3d3e2e417323e8...


`build_run_manifest` assembles all of this into one `RunManifest`, and `write_run_manifest` writes it as deterministic JSON:

In [5]:
from datetime import UTC, datetime

from nuclearff.provenance import build_run_manifest, write_run_manifest

# write_run_manifest creates its target file exclusively (never
# overwrites) -- a fixed run_id would collide the second time this
# notebook is re-run, so it's timestamped instead.
run_id = f"tutorial-demo-{datetime.now(UTC):%Y%m%dT%H%M%SZ}"
manifest = build_run_manifest(run_id=run_id, config=cfg, repo_root=".")
path = write_run_manifest(manifest, "./demo/data/manifests")
path

PosixPath('demo/data/manifests/tutorial-demo-20260922T105122Z.json')

In [6]:
import json

print(json.dumps(json.loads(path.read_text()), indent=2))

{
  "artifact_paths": [],
  "config_hash": "2c3d3e2e417323e85ae5c7789a75066b0bc58a411e9cca4f271a1e7441990dd0",
  "created_at": "2026-09-22T10:51:22.668131Z",
  "git_commit": "d646fcee482c75d1dd16fe3e4cee95782bfcb703",
  "git_dirty": true,
  "package_version": "0.1.0",
  "python_version": "3.14.2",
  "run_id": "tutorial-demo-20260922T105122Z",
  "seed": null,
  "source_hashes": {}
}


`source_hashes` accepts `sha256_file` results for any raw input worth pinning (a downloaded CSV, a hand-authored context-deltas file) alongside the config hash -- enough, together, to answer "what exact code, config, and data produced this artifact" months later. `build_run_manifest` has no CLI command yet; it's a library primitive for a future pipeline to call.

## What's Next

That's every feature this tutorial covers as a Python library. The final notebook, `18_cli_reference.ipynb`, covers the same functionality one more way -- as a single `nuclearff` command-line tool, for when you want a repeatable, scriptable command instead of a Python session.

## See Also

- Every prior notebook -- this one is a capstone over all of them, not new functionality of its own.
- [Querying and Provenance](https://nolmacdonald.github.io/nuclearff/tutorial/17_querying_and_provenance.html) -- the matching docs page.